# ClaimWise Insurance Prediction — 14. Clustering (K-Means, Unsupervised)

## 1. Objective

Unsupervised learning on the actual ClaimWise features: group the 50,000 records by similarity
**without using the target**.

| Rule | How it is respected here |
|---|---|
| `loss` is not a clustering input | `loss` is dropped before fitting (printed below) |
| `loss` is only used afterwards | for *profiling* the found clusters (analysis, not input) |
| cluster count must be justified | elbow curve + silhouette scores, both executed below |
| reproducibility | `random_state=42` everywhere |

Artifacts are written to `Outputs/clustering/`.

## 2. Import Libraries

In [1]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
from pathlib import Path
import sys


def find_project_root(start: Path) -> Path:
    """Walk upwards from the notebook folder until the ClaimWise project is found."""
    for candidate in [start, *start.parents]:
        if (candidate / "Dataset").is_dir() and (candidate / "Src").is_dir():
            return candidate
    raise FileNotFoundError(f"ClaimWise project root not found above: {start}")


PROJECT_ROOT = find_project_root(Path.cwd())
DATASET_DIR = PROJECT_ROOT / "Dataset"
SRC_DIR = PROJECT_ROOT / "Src"
MODELS_DIR = PROJECT_ROOT / "Models"
OUTPUTS_DIR = PROJECT_ROOT / "Outputs"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from academic_common import ensure_dirs, save_figure, style_axes

from academic_common import CLUSTERING_DIR

ensure_dirs()
print("clustering output folder:", CLUSTERING_DIR)

clustering output folder: /Users/padaltiruvinayak/Desktop/Credit ML/Outputs/clustering


In [3]:
# Src/model_training_utils.py switches matplotlib to the non-interactive Agg backend so the
# command-line scripts can run without a display. Switch back to the inline backend here so
# that every figure below is rendered inside this notebook.
%matplotlib inline

## 3. Load the ClaimWise Features (target excluded)

In [4]:
# The real ClaimWise preprocessed data. The target "loss" is REMOVED: unsupervised methods
# must not see the target as an input feature.
df = pd.read_csv(DATASET_DIR / "05_preprocessed_final.csv")
target_present = "loss" in df.columns
features = df.drop(columns=["loss"]) if target_present else df
loss_only = df["loss"] if target_present else None

print("raw preprocessed shape:", df.shape)
print("features used:", features.shape, "| 'loss' excluded from the inputs:", target_present)
print("features numeric:", all(pd.api.types.is_numeric_dtype(d) for d in features.dtypes))
print("missing values:", int(features.isna().sum().sum()))
print("duplicate rows:", int(features.duplicated().sum()))

raw preprocessed shape: (50000, 131)
features used: (50000, 130) | 'loss' excluded from the inputs: True
features numeric: True
missing values: 0
duplicate rows: 0


## 4. Scaling

Centroid-based clustering is distance-driven, so every feature is standardised first
(`StandardScaler`). This is an unsupervised transformation — it is fitted on the feature matrix
itself and never touches the target, so there is no target leakage.

In [5]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
features_scaled = scaler.fit_transform(features)
print("scaled matrix:", features_scaled.shape)
print("max |column mean|:", float(np.abs(features_scaled.mean(axis=0)).max()))

scaled matrix: (50000, 130)
max |column mean|: 4.320099833421409e-16


## 5. Choosing the Number of Clusters (data-driven)

Two standard criteria are computed on a **reproducible 10,000-row sample**
(`random_state=42`) — an honest, documented reduction because k-means cost grows with both rows and
features:

* **Elbow method** — within-cluster sum of inertia for k = 1…10; the "elbow" is where adding a
  cluster stops helping much.
* **Silhouette score** — how similar each record is to its own cluster vs the neighbouring cluster
  (−1 … +1, higher is better), computed for k = 2…8 on a 3,000-row sub-sample to keep the O(n²)
  distance work bounded.

**Selection rule (declared in advance):** pick the k with the **highest silhouette score**; the elbow
plot is reported alongside it as corroboration.

In [6]:
from sklearn.cluster import KMeans

sample_idx = np.random.RandomState(42).choice(len(features_scaled), size=10000, replace=False)
X_sample = features_scaled[sample_idx]
sil_idx = np.random.RandomState(42).choice(len(X_sample), size=3000, replace=False)
X_sil = X_sample[sil_idx]
print("k-search sample:", X_sample.shape, "| silhouette sample:", X_sil.shape)

inertias = {}
for k in range(1, 11):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_sample)
    inertias[k] = float(km.inertia_)
    print(f"k={k:2d} inertia={inertias[k]:,.1f}")

k-search sample: (10000, 130) | silhouette sample: (3000, 130)


k= 1 inertia=1,267,064.9


k= 2 inertia=1,186,135.7


k= 3 inertia=1,153,868.0


k= 4 inertia=1,127,364.0


...

k= 6 inertia=1,084,227.0


k= 7 inertia=1,065,025.5


k= 8 inertia=1,053,594.5


k= 9 inertia=1,038,144.9


k=10 inertia=1,019,529.7


In [7]:
from sklearn.metrics import silhouette_score

silhouettes = {}
for k in range(2, 9):
    km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X_sample)
    score = float(silhouette_score(X_sil, km.labels_[sil_idx], random_state=42))
    silhouettes[k] = score
    print(f"k={k} silhouette={score:.4f}")

best_k = max(silhouettes, key=silhouettes.get)
print(f"\nSelected k = {best_k} (highest silhouette {silhouettes[best_k]:.4f})")

k=2 silhouette=0.0836


k=3 silhouette=0.0816


k=4 silhouette=0.0399


k=5 silhouette=0.0385


k=6 silhouette=0.0336


k=7 silhouette=0.0337


k=8 silhouette=0.0240

Selected k = 2 (highest silhouette 0.0836)


In [8]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

ks = list(inertias)
axes[0].plot(ks, [inertias[k] for k in ks], marker="o", color="#D4512A", linewidth=2)
axes[0].axvline(best_k, linestyle="--", color="#18181B", label=f"selected k = {best_k}")
style_axes(axes[0], "Elbow method — inertia vs k", "k", "inertia (within-cluster sum of squares)")
axes[0].legend()

ks2 = list(silhouettes)
axes[1].bar(ks2, [silhouettes[k] for k in ks2], color="#E58A3A")
axes[1].axvline(best_k, linestyle="--", color="#18181B", linewidth=1.2)
style_axes(axes[1], "Silhouette score vs k (higher is better)", "k", "silhouette score")
for k, v in zip(ks2, [silhouettes[k] for k in ks2]):
    axes[1].text(k, v + 0.004, f"{v:.3f}", ha="center", fontsize=9)

fig.tight_layout()
path = save_figure(fig, CLUSTERING_DIR / "kmeans_elbow_silhouette.png")
print("Saved:", path)

Saved: /Users/padaltiruvinayak/Desktop/Credit ML/Outputs/clustering/kmeans_elbow_silhouette.png


## 6. Final K-Means Model (all 50,000 rows)

The selected k is refitted on the complete scaled feature matrix so every ClaimWise record gets a
cluster label.

In [9]:
from sklearn.metrics import silhouette_score as sil_score

final_km = KMeans(n_clusters=best_k, random_state=42, n_init=10)
cluster_labels = final_km.fit_predict(features_scaled)
print("fitted on:", features_scaled.shape)
print("inertia (full data):", round(final_km.inertia_, 1))

final_sil = float(sil_score(features_scaled, cluster_labels, sample_size=5000, random_state=42))
print("silhouette (5,000-row sample of the full fit):", round(final_sil, 4))

sizes = pd.Series(cluster_labels).value_counts().sort_index()
print("\ncluster sizes:")
print(sizes.to_string())

fitted on: (50000, 130)
inertia (full data): 6093954.6
silhouette (5,000-row sample of the full fit): 0.0822

cluster sizes:
0    21447
1    28553


In [10]:
fig, ax = plt.subplots(figsize=(7.5, 4.6))
bars = ax.bar([f"cluster {i}" for i in sizes.index], sizes.values, color="#D4512A")
style_axes(ax, f"K-Means cluster sizes (k = {best_k})", "", "records")
ax.bar_label(bars, fmt="%d", fontsize=10, padding=2)
path = save_figure(fig, CLUSTERING_DIR / "kmeans_cluster_sizes.png")
print("Saved:", path)

Saved: /Users/padaltiruvinayak/Desktop/Credit ML/Outputs/clustering/kmeans_cluster_sizes.png


## 7. 2D Visualisation (PCA used **only** for plotting)

Clusters live in 130 dimensions. For a picture, the data is projected onto the first two principal
components. This plot is a **visualisation of the labels**, not the space the algorithm worked in —
k-means itself ran on all 130 scaled features.

In [11]:
from sklearn.decomposition import PCA

projection = PCA(n_components=2, random_state=42).fit_transform(features_scaled)
n_show = 12000
show_idx = np.random.RandomState(42).choice(len(projection), size=n_show, replace=False)

fig, ax = plt.subplots(figsize=(9, 6.5))
scatter = ax.scatter(projection[show_idx, 0], projection[show_idx, 1],
                     c=cluster_labels[show_idx], cmap="tab10", s=8, alpha=0.65)
fig.colorbar(scatter, ax=ax, label="cluster")
style_axes(ax, f"K-Means clusters in a 2D PCA projection ({n_show} of {len(projection)} rows)",
           "PC1", "PC2")
path = save_figure(fig, CLUSTERING_DIR / "kmeans_pca_visualization.png")
print("Saved:", path)

Saved: /Users/padaltiruvinayak/Desktop/Credit ML/Outputs/clustering/kmeans_pca_visualization.png


## 8. Cluster Profiling (analysis only)

`loss` was **not** an input. Now — and only now — it is read back to describe what the clusters mean
in terms of the claim amount, which is the academically interesting question: do the found groups
differ in loss?

In [12]:
profile = pd.DataFrame({"cluster": cluster_labels, "loss": loss_only.to_numpy()})
summary = profile.groupby("cluster")["loss"].agg(["count", "mean", "median", "min", "max"])
summary["share_%"] = (100 * summary["count"] / len(profile)).round(2)
summary = summary.round(2)
display(summary)

csv_path = CLUSTERING_DIR / "cluster_profiles.csv"
summary.to_csv(csv_path)
print("Saved:", csv_path)

pd.DataFrame({
    "k": list(silhouettes),
    "silhouette": [round(silhouettes[k], 4) for k in silhouettes],
}).to_csv(CLUSTERING_DIR / "kmeans_silhouette_by_k.csv", index=False)

stats = {
    "selected_k": best_k,
    "silhouette_selected_k": round(silhouettes[best_k], 4),
    "silhouette_full_fit_sample": round(final_sil, 4),
    "inertia_full": round(final_km.inertia_, 1),
    "rows_clustered": int(len(profile)),
}
pd.DataFrame([stats]).to_csv(CLUSTERING_DIR / "kmeans_summary.csv", index=False)
print("Saved:", CLUSTERING_DIR / "kmeans_summary.csv")
print(stats)

,count,mean,median,min,max,share_%
cluster,,,,,,
0,21447,2982.89,2074.83,28.46,79623.52,42.89
1,28553,3074.16,2149.72,10.00,85923.56,57.11


Saved: /Users/padaltiruvinayak/Desktop/Credit ML/Outputs/clustering/cluster_profiles.csv
Saved: /Users/padaltiruvinayak/Desktop/Credit ML/Outputs/clustering/kmeans_summary.csv
{'selected_k': 2, 'silhouette_selected_k': 0.0836, 'silhouette_full_fit_sample': 0.0822, 'inertia_full': 6093954.6, 'rows_clustered': 50000}


## 9. Conclusion

Actual results of this executed run:

* Clustering input: 50,000 × 130 ClaimWise features, **`loss` excluded**.
* Selection: highest silhouette across k = 2…8 → **k = 2** (silhouette
  **0.0836**), corroborated by the elbow plot.
* Full-data fit silhouette (5,000-row sample): **0.0822**.
* Cluster sizes and per-cluster loss statistics are in
  `Outputs/clustering/cluster_profiles.csv`.

Artifacts: `kmeans_elbow_silhouette.png`, `kmeans_cluster_sizes.png`,
`kmeans_pca_visualization.png`, `cluster_profiles.csv`, `kmeans_summary.csv`,
`kmeans_silhouette_by_k.csv`.

The 2D plot is a PCA projection of the labels; the algorithm itself used all 130 features.